# Lựa chọn đặc trưng theo mô hình (Model-based)

Notebook này dùng nhiều mô hình để đo mức quan trọng của đặc trưng và tạo bảng xếp hạng tổng hợp.

## 1. Import thư viện

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)

## 2. Nạp dữ liệu

In [ ]:
duong_dan_ung_vien = [
    Path("../../data/raw/abalone.csv"),
    Path("../data/raw/abalone.csv"),
    Path("data/raw/abalone.csv"),
    Path("AbaloneAge/data/raw/abalone.csv"),
]

duong_dan_du_lieu = None
for p in duong_dan_ung_vien:
    p_day_du = p.resolve()
    if p_day_du.exists():
        duong_dan_du_lieu = p_day_du
        break

if duong_dan_du_lieu is None:
    raise FileNotFoundError(
        "Khong tim thay file abalone.csv. Da thu: "
        + ", ".join(str(p.resolve()) for p in duong_dan_ung_vien)
    )

df = pd.read_csv(duong_dan_du_lieu, header=None)
df.columns = [
    "sex",
    "length",
    "diameter",
    "height",
    "whole_weight",
    "shucked_weight",
    "viscera_weight",
    "shell_weight",
    "rings",
]

print("Duong dan du lieu:", duong_dan_du_lieu)
print("Kich thuoc:", df.shape)
df.head()

## 3. Chia train, validation, test

In [ ]:
X = df.drop(columns=["rings"])
y = df["rings"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 4. Tien xu ly bang Pipeline

In [ ]:
cot_so = [
    "length",
    "diameter",
    "height",
    "whole_weight",
    "shucked_weight",
    "viscera_weight",
    "shell_weight",
]
cot_hang_muc = ["sex"]

bien_doi_so = Pipeline(steps=[
    ("dien_khuyet", SimpleImputer(strategy="median")),
    ("chuan_hoa", StandardScaler()),
])

bien_doi_hang_muc = Pipeline(steps=[
    ("dien_khuyet", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])

tien_xu_ly = ColumnTransformer(transformers=[
    ("so", bien_doi_so, cot_so),
    ("hang_muc", bien_doi_hang_muc, cot_hang_muc),
])

X_train_txl = tien_xu_ly.fit_transform(X_train)
X_val_txl = tien_xu_ly.transform(X_val)
X_test_txl = tien_xu_ly.transform(X_test)

ten_dac_trung = tien_xu_ly.get_feature_names_out()
print("So dac trung sau tien xu ly:", len(ten_dac_trung))

## 5. Huan luyen cac mo hinh va lay do quan trong

In [ ]:
mo_hinh_dict = {
    "random_forest": RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    "extra_trees": ExtraTreesRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    "ridge": Ridge(alpha=1.0),
}

ket_qua_metric = []
tat_ca_importance = []

for ten_mo_hinh, mo_hinh in mo_hinh_dict.items():
    mo_hinh.fit(X_train_txl, y_train)
    du_doan_val = mo_hinh.predict(X_val_txl)

    mae = mean_absolute_error(y_val, du_doan_val)
    rmse = np.sqrt(mean_squared_error(y_val, du_doan_val))
    r2 = r2_score(y_val, du_doan_val)

    ket_qua_metric.append({
        "mo_hinh": ten_mo_hinh,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })

    if hasattr(mo_hinh, "feature_importances_"):
        importance = mo_hinh.feature_importances_
    else:
        importance = np.abs(mo_hinh.coef_)

    bang_importance = pd.DataFrame({
        "dac_trung": ten_dac_trung,
        "importance": importance,
        "mo_hinh": ten_mo_hinh,
    })
    tat_ca_importance.append(bang_importance)

bang_metric = pd.DataFrame(ket_qua_metric).sort_values(by="RMSE")
bang_metric

## 6. Tong hop xep hang dac trung

In [ ]:
bang_all = pd.concat(tat_ca_importance, ignore_index=True)

# Chuan hoa importance theo tung mo hinh de so sanh cong bang
bang_all["importance_chuan_hoa"] = bang_all.groupby("mo_hinh")["importance"].transform(
    lambda x: x / (x.max() + 1e-12)
)

bang_tong_hop = (
    bang_all.groupby("dac_trung")["importance_chuan_hoa"]
    .mean()
    .reset_index()
    .rename(columns={"importance_chuan_hoa": "diem_trung_binh"})
    .sort_values(by="diem_trung_binh", ascending=False)
)

bang_tong_hop.head(15)

## 7. Bieu do top dac trung

In [ ]:
top_n = 12
bang_top = bang_tong_hop.head(top_n).sort_values(by="diem_trung_binh")

plt.figure(figsize=(10, 6))
plt.barh(bang_top["dac_trung"], bang_top["diem_trung_binh"], color="steelblue")
plt.title("Top dac trung quan trong (tong hop model-based)", fontweight="bold")
plt.xlabel("Diem quan trong trung binh (da chuan hoa)")
plt.tight_layout()

duong_dan_hinh = Path("../../outputs/figures").resolve()
duong_dan_hinh.mkdir(parents=True, exist_ok=True)
plt.savefig(duong_dan_hinh / "03_feature_selection_model_based_top_features.png", dpi=300, bbox_inches="tight")
plt.show()

print("Da luu hinh: 03_feature_selection_model_based_top_features.png")

## 8. Danh gia mo hinh tot nhat tren test

In [ ]:
mo_hinh_tot_nhat = bang_metric.iloc[0]["mo_hinh"]
model = mo_hinh_dict[mo_hinh_tot_nhat]

du_doan_test = model.predict(X_test_txl)
mae_test = mean_absolute_error(y_test, du_doan_test)
rmse_test = np.sqrt(mean_squared_error(y_test, du_doan_test))
r2_test = r2_score(y_test, du_doan_test)

print("Mo hinh tot nhat tren validation:", mo_hinh_tot_nhat)
print(f"MAE test : {mae_test:.4f}")
print(f"RMSE test: {rmse_test:.4f}")
print(f"R2 test  : {r2_test:.4f}")

## 9. Luu ket qua

In [ ]:
duong_dan_metrics = Path("../../outputs/metrics").resolve()
duong_dan_metrics.mkdir(parents=True, exist_ok=True)

bang_metric.to_csv(duong_dan_metrics / "03_model_based_model_metrics.csv", index=False)
bang_all.to_csv(duong_dan_metrics / "03_model_based_all_importance.csv", index=False)
bang_tong_hop.to_csv(duong_dan_metrics / "03_model_based_feature_ranking.csv", index=False)

tom_tat = {
    "phuong_phap": "model_based",
    "mo_hinh_tot_nhat": mo_hinh_tot_nhat,
    "test": {
        "MAE": mae_test,
        "RMSE": rmse_test,
        "R2": r2_test,
    },
    "top_10": bang_tong_hop.head(10).to_dict(orient="records"),
}

with open(duong_dan_metrics / "03_model_based_summary.json", "w", encoding="utf-8") as f:
    json.dump(tom_tat, f, ensure_ascii=False, indent=2)

print("Da luu: 03_model_based_model_metrics.csv")
print("Da luu: 03_model_based_all_importance.csv")
print("Da luu: 03_model_based_feature_ranking.csv")
print("Da luu: 03_model_based_summary.json")